# 第8章: ニューラルネット

第7章で取り組んだポジネガ分類を題材として、ニューラルネットワークで分類モデルを実装する。なお、この章ではPyTorchやTensorFlow、JAXなどの深層学習フレームワークを活用せよ。

In [1]:
pip install gensim

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.6/26.6 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 114.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.2/38.2 MB 16.5 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.1
    Uninstalling scipy-1.16.1:
      Successfully uninstalled scipy-1.16.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tsfresh 0.21.0 requires scipy>=1.14.0; python_version >= "3.10", but you have scipy 1.13.1 which is incompatible.
thinc 8.3.6 re

In [1]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


## 70. 単語埋め込みの読み込み

事前学習済み単語埋め込みを活用し、$|V| \times d_\rm{emb}$ の単語埋め込み行列$\pmb{E}$を作成せよ。ここで、$|V|$は単語埋め込みの語彙数、$d_\rm{emb}$は単語埋め込みの次元数である。ただし、単語埋め込み行列の先頭の行ベクトル$\pmb{E}_{0,:}$は、将来的にパディング（`<PAD>`）トークンの埋め込みベクトルとして用いたいので、ゼロベクトルとして予約せよ。ゆえに、$\pmb{E}$の2行目以降に事前学習済み単語埋め込みを読み込むことになる。

もし、Google Newsデータセットの[学習済み単語ベクトル](https://drive.google.com/file/d/0B7XkCwpI5KDYNlNUTTlSS21pQmM/edit?usp=sharing)（300万単語・フレーズ、300次元）を全て読み込んだ場合、$|V|=3000001, d_\rm{emb}=300$になるはずである（ただ、300万単語の中には、殆ど用いられない稀な単語も含まれるので、語彙を削減した方がメモリの節約になる）。

また、単語埋め込み行列の構築と同時に、単語埋め込み行列の各行のインデックス番号（トークンID）と、単語（トークン）への双方向の対応付けを保持せよ。

In [2]:
from gensim.models import KeyedVectors
import numpy as np

In [ ]:
def main():
  model = KeyedVectors.load_word2vec_format('/content/drive/MyDrive/nlp100/lesson08/GoogleNews-vectors-negative300.bin.gz', binary = True)

  vocab_size = len(model.key_to_index)
  embedding_dim = model.vector_size
  embedding_matrix = np.zeros((vocab_size + 1, embedding_dim))

  word_to_id = {"<PAD>": 0}
  id_to_word = {0: "<PAD>"}

  for i, word in enumerate(model.key_to_index, start = 1):
    embedding_matrix[i] = model[word]
    word_to_id[word] = i
    id_to_word[i] = word

  print(len(embedding_matrix))
  print(embedding_matrix.shape[1])

In [ ]:
main()

3000001
300


## 71. データセットの読み込み

[General Language Understanding Evaluation (GLUE)](https://gluebenchmark.com/) ベンチマークで配布されている[Stanford Sentiment Treebank (SST)](https://dl.fbaipublicfiles.com/glue/data/SST-2.zip) をダウンロードし、訓練セット（train.tsv）と開発セット（dev.tsv）のテキストと極性ラベルと読み込み、全てのテキストをトークンID列に変換せよ。このとき、単語埋め込みの語彙でカバーされていない単語は無視し、トークン列に含めないことにせよ。また、テキストの全トークンが単語埋め込みの語彙に含まれておらず、空のトークン列となってしまう事例は、訓練セットおよび開発セットから削除せよ（このため、第7章の実験で得られた正解率と比較できなくなることに注意せよ）。

事例の表現方法は任意でよいが、例えば"contains no wit , only labored gags"がネガティブに分類される事例は、次のような辞書オブジェクトで表現すればよい。

```
{'text': 'contains no wit , only labored gags',
 'label': tensor([0.]),
 'input_ids': tensor([ 3475,    87, 15888,    90, 27695, 42637])}
```

この例では、`text`はテキスト、`label`は分類ラベル（ポジティブなら`tensor([1.])`、ネガティブなら`tensor([0.])`）、`input_ids`はテキストのトークン列をID列で表現している。

In [3]:
import torch
import pandas as pd
from typing import Set, Dict, List

In [4]:
train_data = pd.read_csv('/content/drive/MyDrive/nlp100/lesson08/SST-2/train.tsv', sep = '\t')
dev_data = pd.read_csv('/content/drive/MyDrive/nlp100/lesson08/SST-2/dev.tsv', sep = '\t')

In [ ]:
# vocabulary
# word_to_id
# text_to_tokens

In [5]:
def get_vocabulary(df: pd.DataFrame) -> Set[str]:
  vocabulary = set()
  for text in df['sentence']:
    vocabulary.update(text.lower().split()) # 文を全て小文字に変換し、空白文字で分割 -> vocabularyに追加

  return vocabulary

def load_word_embedding(model_path: str, vocabulary: Set[str]) -> Dict[str, int]:
  word_to_id = {"<PAD>": 0}
  model = KeyedVectors.load_word2vec_format(model_path, binary = True)
  for word in vocabulary:
    if word in model.key_to_index:
      word_to_id[word] = len(word_to_id) # 単語を対応するIDに置き換えるための辞書

  return word_to_id

def tokenize_text(text: str, word_to_id: Dict[str, int]) -> List[int]:
  words = text.lower().split()
  ids = [word_to_id[word] for word in words if word in word_to_id] # 文の各単語において、word_to_idに単語が存在していれば、そのIDをidsに追加

  return ids

def tokenize_df(df: pd.DataFrame, word_to_id: Dict[str, int]) -> List[Dict]:
  tokenized_text = []
  for _, row in df.iterrows():
    ids = tokenize_text(row['sentence'], word_to_id)

    if not ids:
      continue

    data = {
        'text': row['sentence'],
        'label': torch.tensor([float(row['label'])]),
        'input_ids': torch.tensor(ids)
    }
    tokenized_text.append(data)

  return tokenized_text

In [ ]:
def main():
  vocabulary = get_vocabulary(train_data)
  vocabulary.update(get_vocabulary(dev_data))

  word_to_id = load_word_embedding('/content/drive/MyDrive/nlp100/lesson08/GoogleNews-vectors-negative300.bin.gz', vocabulary)

  train_data_tokenized = tokenize_df(train_data, word_to_id)
  dev_data_tokenized = tokenize_df(dev_data, word_to_id)

  sample = train_data_tokenized[0]
  print(sample)

In [ ]:
main()

{'text': 'hide new secretions from the parental units ', 'label': tensor([0.]), 'input_ids': tensor([12350, 12126,  3677,  4648, 10836,  4167,  8417])}


## 72. Bag of wordsモデルの構築

単語埋め込みの平均ベクトルでテキストの特徴ベクトルを表現し、重みベクトルとの内積でポジティブ及びネガティブを分類するニューラルネットワーク（ロジスティック回帰モデル）を設計せよ。

In [6]:
import torch.nn as nn

In [7]:
class MeanEmbeddingClassifier(nn.Module):
  def __init__(self, embedding_dim):
    super().__init__()
    self.linear = nn.Linear(in_features = embedding_dim, out_features = 1) # 全結合層, 入力: 埋め込みモデルの次元数, 出力: ネガ or ポジ
    self.sigmoid = nn.Sigmoid() # 活性化関数にシグモイド関数(ロジスティック回帰モデル)

  def forward(self, x): # 順伝播
    output = self.sigmoid(self.linear(x)) # シグモイド関数で値を0~1に変換
    return output

In [8]:
def load_word_embedding(model_path: str, vocabulary: Set[str]) -> tuple[Dict[str, int], torch.Tensor]:
  word_to_id = {"<PAD>": 0}
  model = KeyedVectors.load_word2vec_format(model_path, binary = True)

  embeddings = [torch.zeros(model.vector_size)] # <PAD>のベクトル
  for word in vocabulary:
    if word in model.key_to_index:
      word_to_id[word] = len(word_to_id)
      embeddings.append(torch.tensor(model[word])) # 埋め込みベクトルをtensorに変換
  embedding_matrix = torch.stack(embeddings) # 0 -> 単語の数, 1 -> 埋め込みの次元数

  return word_to_id, embedding_matrix

def create_mean_embedding_features(data: List[Dict], embedding_matrix: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
  features = []
  labels = []
  for item in data:
    input_ids = item['input_ids'] # トークンID
    embeddings = embedding_matrix[input_ids] # トークンIDに対応する埋め込みを取得
    mean_embedding = torch.mean(embeddings, dim = 0)
    features.append(mean_embedding)
    labels.append(item['label'])

  return torch.stack(features), torch.cat(labels) # featuresもcatにすると、各文を区別できない

In [ ]:
def main():
  # データの前処理
  vocabulary = get_vocabulary(train_data)
  vocabulary.update(get_vocabulary(dev_data))

  word_to_id, embedding_matrix = load_word_embedding('/content/drive/MyDrive/nlp100/lesson08/GoogleNews-vectors-negative300.bin.gz', vocabulary)

  train_data_tokenized = tokenize_df(train_data, word_to_id)
  dev_data_tokenized = tokenize_df(dev_data, word_to_id)
  X_train, y_train = create_mean_embedding_features(train_data_tokenized, embedding_matrix)
  X_dev, y_dev = create_mean_embedding_features(dev_data_tokenized, embedding_matrix)

  model = MeanEmbeddingClassifier(embedding_matrix.size(1))

  print(f'モデル構造: {model}')
  print(f'特徴ベクトルの形状: {X_train.shape}')
  print(f'正解ラベルの形状: {y_train.shape}')


In [ ]:
main()

モデル構造: MeanEmbeddingClassifier(
  (linear): Linear(in_features=300, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)
特徴ベクトルの形状: torch.Size([66650, 300])
正解ラベルの形状: torch.Size([66650])


## 73. モデルの学習

問題72で設計したモデルの重みベクトルを訓練セット上で学習せよ。ただし、学習中は単語埋め込み行列の値を固定せよ（単語埋め込み行列のファインチューニングは行わない）。また、学習時に損失値を表示するなど、学習の進捗状況をモニタリングできるようにせよ。

In [9]:
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
def train_model():
  # データの前処理
  vocabulary = get_vocabulary(train_data)
  vocabulary.update(get_vocabulary(dev_data))

  word_to_id, embedding_matrix = load_word_embedding('/content/drive/MyDrive/nlp100/lesson08/GoogleNews-vectors-negative300.bin.gz', vocabulary)

  train_data_tokenized = tokenize_df(train_data, word_to_id)
  dev_data_tokenized = tokenize_df(dev_data, word_to_id)
  X_train, y_train = create_mean_embedding_features(train_data_tokenized, embedding_matrix)
  X_dev, y_dev = create_mean_embedding_features(dev_data_tokenized, embedding_matrix)

  # データセットの構築
  train_dataset = TensorDataset(X_train, y_train)
  train_loader = DataLoader(train_dataset, batch_size = 16, shuffle = True)
  # dev_dataset = TensorDataset(X_dev, y_dev)
  # dev_loader = DataLoader(dev_dataset, batch_size = 100, shuffle = True)

  # モデル構築
  model = MeanEmbeddingClassifier(embedding_matrix.size(1))

  # モデルの学習
  num_epoches = 15
  criterion = nn.CrossEntropyLoss() # criterion = nn.BCELoss()
  optimizer = optim.Adam(model.parameters(), lr = 0.001)
  # accs = []
  # losses[]
  for epoch in range(num_epoches):
    model.train()
    running_loss = 0.0
    running_acc = 0.0
    for inputs, labels in train_loader:
      optimizer.zero_grad() # 勾配を初期化
      output = model(inputs) # 順伝播
      loss = criterion(output.squeeze(), labels.float())
      loss.backward() # 誤差逆伝播
      optimizer.step() # 重み・バイアスの更新

      running_loss += loss.item()
      pred = (output.squeeze() > 0.5).float()
      running_acc += torch.mean(pred.eq(labels).float()) # pred.eq(labels)はpredとlabelsが等しいかどうか

    running_loss /= len(train_loader)
    running_acc /= len(train_loader)
    # losses.append(running_loss)
    # accs.append(running_acc.cpu())
    print(f"epoch: {epoch}, loss: {running_loss}, acc: {running_acc}")

  return model

In [ ]:
model = train_model()

epoch: 0, loss: 23.796255561175737, acc: 0.81673663854599
epoch: 1, loss: 23.40306407742585, acc: 0.8319101333618164
epoch: 2, loss: 23.31097388576557, acc: 0.8364108204841614
epoch: 3, loss: 23.264050347840506, acc: 0.838772177696228
epoch: 4, loss: 23.22903753906503, acc: 0.8410225510597229
epoch: 5, loss: 23.207554874257635, acc: 0.8421657085418701
epoch: 6, loss: 23.195641674448307, acc: 0.8429758548736572
epoch: 7, loss: 23.186330023603947, acc: 0.843699038028717
epoch: 8, loss: 23.175534256707575, acc: 0.8450011610984802
epoch: 9, loss: 23.16505895368308, acc: 0.8454962372779846
epoch: 10, loss: 23.162504047827827, acc: 0.8456313014030457
epoch: 11, loss: 23.15690244623405, acc: 0.845991313457489
epoch: 12, loss: 23.149282634172426, acc: 0.8465134501457214
epoch: 13, loss: 23.142721837950738, acc: 0.8464654088020325
epoch: 14, loss: 23.14399148143946, acc: 0.8470385074615479


## 74. モデルの評価

問題73で学習したモデルの開発セットにおける正解率を求めよ。

In [10]:
def calculate_accuracy(model, X, y):
  model.eval()
  with torch.no_grad():
    output = model(X)
    pred = (output.squeeze() > 0.5).float()
    accuracy = torch.mean(pred.eq(y.float()).float())

  print(f"正解率: {accuracy}")

In [ ]:
def evaluate_model(model):
  # データの前処理
  vocabulary = get_vocabulary(train_data)
  vocabulary.update(get_vocabulary(dev_data))

  word_to_id, embedding_matrix = load_word_embedding('/content/drive/MyDrive/nlp100/lesson08/GoogleNews-vectors-negative300.bin.gz', vocabulary)

  dev_data_tokenized = tokenize_df(dev_data, word_to_id)
  X_dev, y_dev = create_mean_embedding_features(dev_data_tokenized, embedding_matrix)

  calculate_accuracy(model, X_dev, y_dev)


In [ ]:
evaluate_model(model)

正解率: 0.8084862232208252


## 75. パディング

複数の事例が与えられたとき、これらをまとめて一つのテンソル・オブジェクトで表現する関数`collate`を実装せよ。与えられた複数の事例のトークン列の長さが異なるときは、トークン列の長さが最も長いものに揃え、0番のトークンIDでパディングをせよ。さらに、トークン列の長さが長いものから順に、事例を並び替えよ。

例えば、訓練データセットの冒頭の4事例が次のように表されているとき、

```
[{'text': 'hide new secretions from the parental units',
  'label': tensor([0.]),
  'input_ids': tensor([  5785,     66, 113845,     18,     12,  15095,   1594])},
 {'text': 'contains no wit , only labored gags',
  'label': tensor([0.]),
  'input_ids': tensor([ 3475,    87, 15888,    90, 27695, 42637])},
 {'text': 'that loves its characters and communicates something rather beautiful about human nature',
  'label': tensor([1.]),
  'input_ids': tensor([    4,  5053,    45,  3305, 31647,   348,   904,  2815,    47,  1276,  1964])},
 {'text': 'remains utterly satisfied to remain the same throughout',
  'label': tensor([0.]),
  'input_ids': tensor([  987, 14528,  4941,   873,    12,   208,   898])}]
```

`collate`関数を通した結果は以下のようになることが想定される。

```
{'input_ids': tensor([
    [     4,   5053,     45,   3305,  31647,    348,    904,   2815,     47,   1276,   1964],
    [  5785,     66, 113845,     18,     12,  15095,   1594,      0,      0,      0,      0],
    [   987,  14528,   4941,    873,     12,    208,    898,      0,      0,      0,      0],
    [  3475,     87,  15888,     90,  27695,  42637,      0,      0,      0,      0,      0]]),
 'label': tensor([
    [1.],
    [0.],
    [0.],
    [0.]])}
```


In [11]:
def collate(data: List[Dict]) -> Dict[str, torch.Tensor]:
  max_tokens_length = max(len(item['input_ids']) for item in data)
  data_size = len(data)

  input_tensor = torch.zeros((data_size, max_tokens_length), dtype = torch.long)
  label_tensor = torch.zeros((data_size, 1), dtype = torch.float)

  lengths = [len(item['input_ids']) for item in data]
  sorted_indices = sorted(range(data_size), key = lambda i : lengths[i], reverse = True)

  for i, idx in enumerate(sorted_indices):
    item = data[idx]
    input_tensor[i, :len(item['input_ids'])] = item['input_ids']
    label_tensor[i] = item['label']

  return {'input_ids': input_tensor, 'label': label_tensor}


In [ ]:
def main():
  data = [{'text': 'hide new secretions from the parental units',
  'label': torch.tensor([0.]),
  'input_ids': torch.tensor([  5785,     66, 113845,     18,     12,  15095,   1594])},
 {'text': 'contains no wit , only labored gags',
  'label': torch.tensor([0.]),
  'input_ids': torch.tensor([ 3475,    87, 15888,    90, 27695, 42637])},
 {'text': 'that loves its characters and communicates something rather beautiful about human nature',
  'label': torch.tensor([1.]),
  'input_ids': torch.tensor([    4,  5053,    45,  3305, 31647,   348,   904,  2815,    47,  1276,  1964])},
 {'text': 'remains utterly satisfied to remain the same throughout',
  'label': torch.tensor([0.]),
  'input_ids': torch.tensor([  987, 14528,  4941,   873,    12,   208,   898])}]

  result = collate(data)
  print(result['input_ids'])
  print(result['label'])

In [ ]:
main()

tensor([[     4,   5053,     45,   3305,  31647,    348,    904,   2815,     47,
           1276,   1964],
        [  5785,     66, 113845,     18,     12,  15095,   1594,      0,      0,
              0,      0],
        [   987,  14528,   4941,    873,     12,    208,    898,      0,      0,
              0,      0],
        [  3475,     87,  15888,     90,  27695,  42637,      0,      0,      0,
              0,      0]])
tensor([[1.],
        [0.],
        [0.],
        [0.]])


## 76. ミニバッチ学習

問題75のパディングの処理を活用して、ミニバッチでモデルを学習せよ。また、学習したモデルの開発セットにおける正解率を求めよ。

In [13]:
from torch.utils.data import Dataset

In [14]:
class SST2Dataset(Dataset):
    def __init__(self, data: List[Dict], embedding_matrix: torch.Tensor):
        self.data = data
        self.embedding_matrix = embedding_matrix

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int) -> Dict:
        return self.data[idx]

In [ ]:
def main():
  # データの前処理
  vocabulary = get_vocabulary(train_data)
  vocabulary.update(get_vocabulary(dev_data))
  word_to_id, embedding_matrix = load_word_embedding('/content/drive/MyDrive/nlp100/lesson08/GoogleNews-vectors-negative300.bin.gz', vocabulary)
  train_data_tokenized = tokenize_df(train_data, word_to_id)
  dev_data_tokenized = tokenize_df(dev_data, word_to_id)

  # データセットの構築
  train_dataset = SST2Dataset(train_data_tokenized, embedding_matrix)
  dev_dataset = SST2Dataset(dev_data_tokenized, embedding_matrix)
  train_loader = DataLoader(train_dataset, batch_size = 16, shuffle = True, collate_fn = collate)
  dev_loader = DataLoader(dev_dataset, batch_size = 16, shuffle = True, collate_fn = collate)

  # モデル構築
  model = MeanEmbeddingClassifier(embedding_matrix.size(1))

  # モデルの学習
  num_epoches = 15
  criterion = nn.BCELoss()
  optimizer = optim.Adam(model.parameters(), lr = 0.001)

  for epoch in range(num_epoches):
    model.train()
    running_loss = 0.0
    running_acc = 0.0
    for batch in train_loader:
      labels = batch['label']
      mean_embeddings = torch.mean(embedding_matrix[batch['input_ids']], dim = 1)
      optimizer.zero_grad()
      output = model(mean_embeddings)

      loss = criterion(output.squeeze(), labels.squeeze().float())
      loss.backward()
      optimizer.step()

      running_loss += loss.item()
      pred = (output.squeeze() > 0.5).float()
      running_acc += (pred == labels.squeeze().float()).sum().item() / labels.size(0)

    running_loss /= len(train_loader)
    running_acc /= len(train_loader)
    print(f"epoch: {epoch + 1}, loss: {running_loss:.4f}, acc: {running_acc:.4f}")

  # 評価
  model.eval()
  total_correct = 0
  total_samples = 0
  with torch.no_grad():
    for batch in dev_loader:
      labels = batch['label']
      mean_embeddings = torch.mean(embedding_matrix[batch['input_ids']], dim=1)

      output = model(mean_embeddings)
      pred = (output.squeeze() > 0.5).float()
      total_samples += labels.size(0)
      total_correct += (pred == labels.squeeze().float()).sum().item()

  accuracy = 100 * total_correct / total_samples
  print(f"\nAccuracy on Dev Data: {accuracy:.2f}%")

In [ ]:
main()

epoch: 1, loss: 0.5859, acc: 0.7287
epoch: 2, loss: 0.4904, acc: 0.8078
epoch: 3, loss: 0.4536, acc: 0.8239
epoch: 4, loss: 0.4356, acc: 0.8300
epoch: 5, loss: 0.4240, acc: 0.8338
epoch: 6, loss: 0.4171, acc: 0.8356
epoch: 7, loss: 0.4112, acc: 0.8377
epoch: 8, loss: 0.4075, acc: 0.8384
epoch: 9, loss: 0.4038, acc: 0.8385
epoch: 10, loss: 0.4015, acc: 0.8403
epoch: 11, loss: 0.3995, acc: 0.8404
epoch: 12, loss: 0.3983, acc: 0.8415
epoch: 13, loss: 0.3963, acc: 0.8412
epoch: 14, loss: 0.3950, acc: 0.8420
epoch: 15, loss: 0.3939, acc: 0.8425

Accuracy on Dev Data: 79.59%


## 77. GPU上での学習

問題76のモデル学習をGPU上で実行せよ。また、学習したモデルの開発セットにおける正解率を求めよ。

In [17]:
def main():
  device = 'cuda' if torch.cuda.is_available() else 'cpu'
  print(device)

  # データの前処理
  vocabulary = get_vocabulary(train_data)
  vocabulary.update(get_vocabulary(dev_data))
  word_to_id, embedding_matrix = load_word_embedding('/content/drive/MyDrive/nlp100/lesson08/GoogleNews-vectors-negative300.bin.gz', vocabulary)
  embedding_matrix = embedding_matrix.to(device)
  train_data_tokenized = tokenize_df(train_data, word_to_id)
  dev_data_tokenized = tokenize_df(dev_data, word_to_id)

  # データセットの構築
  train_dataset = SST2Dataset(train_data_tokenized, embedding_matrix)
  dev_dataset = SST2Dataset(dev_data_tokenized, embedding_matrix)
  train_loader = DataLoader(train_dataset, batch_size = 16, shuffle = True, collate_fn = collate)
  dev_loader = DataLoader(dev_dataset, batch_size = 16, shuffle = True, collate_fn = collate)

  # モデル構築
  model = MeanEmbeddingClassifier(embedding_matrix.size(1))
  model.to(device)

  # モデルの学習
  num_epoches = 15
  criterion = nn.BCELoss()
  optimizer = optim.Adam(model.parameters(), lr = 0.001)

  for epoch in range(num_epoches):
    model.train()
    running_loss = 0.0
    running_acc = 0.0
    for batch in train_loader:
      input_ids = batch['input_ids'].to(device)
      labels = batch['label'].to(device)
      mean_embeddings = torch.mean(embedding_matrix[input_ids], dim = 1)
      optimizer.zero_grad()
      output = model(mean_embeddings)

      loss = criterion(output.squeeze(), labels.squeeze().float())
      loss.backward()
      optimizer.step()

      running_loss += loss.item()
      pred = (output.squeeze() > 0.5).float()
      running_acc += (pred == labels.squeeze().float()).sum().item() / labels.size(0)

    running_loss /= len(train_loader)
    running_acc /= len(train_loader)
    print(f"epoch: {epoch + 1}, loss: {running_loss:.4f}, acc: {running_acc:.4f}")

  # 評価
  model.eval()
  total_correct = 0
  total_samples = 0
  with torch.no_grad():
    for batch in dev_loader:
      labels = batch['label']
      mean_embeddings = torch.mean(embedding_matrix[batch['input_ids']], dim=1)
      mean_embeddings = mean_embeddings.to(device)
      labels = labels.to(device)
      output = model(mean_embeddings)

      pred = (output.squeeze() > 0.5).float()
      total_samples += labels.size(0)
      total_correct += (pred == labels.squeeze().float()).sum().item()

  accuracy = 100 * total_correct / total_samples
  print(f"\nAccuracy on Dev Data: {accuracy:.2f}%")

In [18]:
main()

cuda
epoch: 1, loss: 0.5850, acc: 0.7217
epoch: 2, loss: 0.4898, acc: 0.8064
epoch: 3, loss: 0.4537, acc: 0.8237
epoch: 4, loss: 0.4354, acc: 0.8309
epoch: 5, loss: 0.4240, acc: 0.8329
epoch: 6, loss: 0.4170, acc: 0.8354
epoch: 7, loss: 0.4124, acc: 0.8374
epoch: 8, loss: 0.4081, acc: 0.8385
epoch: 9, loss: 0.4051, acc: 0.8391
epoch: 10, loss: 0.4017, acc: 0.8400
epoch: 11, loss: 0.3995, acc: 0.8405
epoch: 12, loss: 0.3978, acc: 0.8412
epoch: 13, loss: 0.3970, acc: 0.8411
epoch: 14, loss: 0.3949, acc: 0.8417
epoch: 15, loss: 0.3939, acc: 0.8420

Accuracy on Dev Data: 79.47%


## 78. 単語埋め込みのファインチューニング

問題77の学習において、単語埋め込みのパラメータも同時に更新するファインチューニングを導入せよ。また、学習したモデルの開発セットにおける正解率を求めよ。

In [19]:
class MeanEmbeddingClassifier(nn.Module):
  def __init__(self, embedding_matrix):
    super().__init__()
    # nn.Embedding
    # from_pretrained を使い、事前学習済みの重みで初期化する
    # freeze = False にすることで、このレイヤーの重みが訓練中に更新される（ファインチューニング）
    self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze = False)
    self.linear = nn.Linear(in_features = embedding_matrix.size(1), out_features = 1) # 全結合層, 入力: 埋め込みモデルの次元数, 出力: ネガ or ポジ
    self.sigmoid = nn.Sigmoid() # 活性化関数にシグモイド関数(ロジスティック回帰モデル)

  def forward(self, x: torch.Tensor): # 順伝播
    embedded = self.embedding(x)
    mean_embedded = torch.mean(embedded, dim = 1)
    output = self.sigmoid(self.linear(mean_embedded)) # シグモイド関数で値を0~1に変換
    return output

In [20]:
def main():
  device = 'cuda' if torch.cuda.is_available() else 'cpu'
  print(device)

  # データの前処理
  vocabulary = get_vocabulary(train_data)
  vocabulary.update(get_vocabulary(dev_data))
  word_to_id, embedding_matrix = load_word_embedding('/content/drive/MyDrive/nlp100/lesson08/GoogleNews-vectors-negative300.bin.gz', vocabulary)
  embedding_matrix = embedding_matrix.to(device)
  train_data_tokenized = tokenize_df(train_data, word_to_id)
  dev_data_tokenized = tokenize_df(dev_data, word_to_id)

  # データセットの構築
  train_dataset = SST2Dataset(train_data_tokenized, embedding_matrix)
  dev_dataset = SST2Dataset(dev_data_tokenized, embedding_matrix)
  train_loader = DataLoader(train_dataset, batch_size = 16, shuffle = True, collate_fn = collate)
  dev_loader = DataLoader(dev_dataset, batch_size = 16, shuffle = True, collate_fn = collate)

  # モデル構築
  model = MeanEmbeddingClassifier(embedding_matrix)
  model.to(device)

  # モデルの学習
  num_epoches = 15
  criterion = nn.BCELoss()
  optimizer = optim.Adam(model.parameters(), lr = 0.001)

  for epoch in range(num_epoches):
    model.train()
    running_loss = 0.0
    running_acc = 0.0
    for batch in train_loader:
      input_ids = batch['input_ids'].to(device)
      labels = batch['label'].to(device)
      optimizer.zero_grad()
      output = model(input_ids)

      loss = criterion(output.squeeze(), labels.squeeze().float())
      loss.backward()
      optimizer.step()

      running_loss += loss.item()
      pred = (output.squeeze() > 0.5).float()
      running_acc += (pred == labels.squeeze().float()).sum().item() / labels.size(0)

    running_loss /= len(train_loader)
    running_acc /= len(train_loader)
    print(f"epoch: {epoch + 1}, loss: {running_loss:.4f}, acc: {running_acc:.4f}")

  # 評価
  model.eval()
  total_correct = 0
  total_samples = 0
  with torch.no_grad():
    for batch in dev_loader:
      input_ids = batch['input_ids'].to(device)
      labels = batch['label'].to(device)
      output = model(input_ids)

      pred = (output.squeeze() > 0.5).float()
      total_samples += labels.size(0)
      total_correct += (pred == labels.squeeze().float()).sum().item()

  accuracy = 100 * total_correct / total_samples
  print(f"\nAccuracy on Dev Data: {accuracy:.2f}%")

In [21]:
main()

cuda
epoch: 1, loss: 0.3831, acc: 0.8332
epoch: 2, loss: 0.2387, acc: 0.9103
epoch: 3, loss: 0.2089, acc: 0.9213
epoch: 4, loss: 0.1925, acc: 0.9279
epoch: 5, loss: 0.1814, acc: 0.9303
epoch: 6, loss: 0.1742, acc: 0.9347
epoch: 7, loss: 0.1698, acc: 0.9363
epoch: 8, loss: 0.1654, acc: 0.9370
epoch: 9, loss: 0.1625, acc: 0.9387
epoch: 10, loss: 0.1592, acc: 0.9401
epoch: 11, loss: 0.1571, acc: 0.9403
epoch: 12, loss: 0.1557, acc: 0.9411
epoch: 13, loss: 0.1535, acc: 0.9425
epoch: 14, loss: 0.1515, acc: 0.9428
epoch: 15, loss: 0.1501, acc: 0.9435

Accuracy on Dev Data: 78.21%


## 79. アーキテクチャの変更

ニューラルネットワークのアーキテクチャを自由に変更し、モデルを学習せよ。また、学習したモデルの開発セットにおける正解率を求めよ。例えば、テキストの特徴ベクトル（単語埋め込みの平均ベクトル）に対して多層のニューラルネットワークを通したり、畳み込みニューラルネットワーク（CNN; Convolutional Neural Network）や再帰型ニューラルネットワーク（RNN; Recurrent Neural Network）などのモデルの学習に挑戦するとよい。

In [22]:
class MeanEmbeddingClassifier(nn.Module):
  def __init__(self, embedding_matrix):
    super().__init__()
    self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze = False)
    self.classifier = nn.Sequential(
            nn.Linear(in_features = embedding_matrix.size(1), out_features = 100),
            nn.ReLU(inplace = True), # inplaceはReLUを実行した後にその値で元の配列を置き換える => メモリ節約
            nn.Linear(in_features = 100, out_features = 50),
            nn.ReLU(inplace = True),
            nn.Linear(in_features = 50, out_features = 10),
            nn.ReLU(inplace = True),
            nn.Linear(in_features = 10, out_features = 1)
        )
    self.sigmoid = nn.Sigmoid()

  def forward(self, x: torch.Tensor):
    embedded = self.embedding(x)
    mean_embedded = torch.mean(embedded, dim = 1)
    # output = self.sigmoid(self.linear(mean_embedded))
    output = self.sigmoid(self.classifier(mean_embedded))
    return output

In [23]:
def main():
  device = 'cuda' if torch.cuda.is_available() else 'cpu'
  print(device)

  # データの前処理
  vocabulary = get_vocabulary(train_data)
  vocabulary.update(get_vocabulary(dev_data))
  word_to_id, embedding_matrix = load_word_embedding('/content/drive/MyDrive/nlp100/lesson08/GoogleNews-vectors-negative300.bin.gz', vocabulary)
  embedding_matrix = embedding_matrix.to(device)
  train_data_tokenized = tokenize_df(train_data, word_to_id)
  dev_data_tokenized = tokenize_df(dev_data, word_to_id)

  # データセットの構築
  train_dataset = SST2Dataset(train_data_tokenized, embedding_matrix)
  dev_dataset = SST2Dataset(dev_data_tokenized, embedding_matrix)
  train_loader = DataLoader(train_dataset, batch_size = 16, shuffle = True, collate_fn = collate)
  dev_loader = DataLoader(dev_dataset, batch_size = 16, shuffle = True, collate_fn = collate)

  # モデル構築
  model = MeanEmbeddingClassifier(embedding_matrix)
  model.to(device)

  # モデルの学習
  num_epoches = 15
  criterion = nn.BCELoss()
  optimizer = optim.Adam(model.parameters(), lr = 0.001)

  for epoch in range(num_epoches):
    model.train()
    running_loss = 0.0
    running_acc = 0.0
    for batch in train_loader:
      input_ids = batch['input_ids'].to(device)
      labels = batch['label'].to(device)
      optimizer.zero_grad()
      output = model(input_ids)

      loss = criterion(output.squeeze(), labels.squeeze().float())
      loss.backward()
      optimizer.step()

      running_loss += loss.item()
      pred = (output.squeeze() > 0.5).float()
      running_acc += (pred == labels.squeeze().float()).sum().item() / labels.size(0)

    running_loss /= len(train_loader)
    running_acc /= len(train_loader)
    print(f"epoch: {epoch + 1}, loss: {running_loss:.4f}, acc: {running_acc:.4f}")

  # 評価
  model.eval()
  total_correct = 0
  total_samples = 0
  with torch.no_grad():
    for batch in dev_loader:
      input_ids = batch['input_ids'].to(device)
      labels = batch['label'].to(device)
      output = model(input_ids)

      pred = (output.squeeze() > 0.5).float()
      total_samples += labels.size(0)
      total_correct += (pred == labels.squeeze().float()).sum().item()

  accuracy = 100 * total_correct / total_samples
  print(f"\nAccuracy on Dev Data: {accuracy:.2f}%")

In [24]:
main()

cuda
epoch: 1, loss: 0.3104, acc: 0.8657
epoch: 2, loss: 0.1955, acc: 0.9233
epoch: 3, loss: 0.1507, acc: 0.9409
epoch: 4, loss: 0.1202, acc: 0.9528
epoch: 5, loss: 0.1030, acc: 0.9588
epoch: 6, loss: 0.0905, acc: 0.9638
epoch: 7, loss: 0.0787, acc: 0.9674
epoch: 8, loss: 0.0732, acc: 0.9704
epoch: 9, loss: 0.0663, acc: 0.9727
epoch: 10, loss: 0.0620, acc: 0.9740
epoch: 11, loss: 0.0567, acc: 0.9762
epoch: 12, loss: 0.0536, acc: 0.9774
epoch: 13, loss: 0.0508, acc: 0.9783
epoch: 14, loss: 0.0481, acc: 0.9792
epoch: 15, loss: 0.0455, acc: 0.9804

Accuracy on Dev Data: 79.13%
